# Reproducing Lee et al. (2014) from CRCNS hc-10

**Paper.** Lee S-H, Marchionni I, Bezaire M, Varga C, Danielson N, Lovett-Barron M, Losonczy A,
Soltesz I (2014). Parvalbumin-positive basket cells differentiate among hippocampal pyramidal
cells. *Neuron* 82:1129-1144.
[doi:10.1016/j.neuron.2014.03.034](https://doi.org/10.1016/j.neuron.2014.03.034)

**Dataset.** CRCNS hc-10, Lee, Krook-Magnuson, Bezaire & Soltesz (2016), *Intracellular, in
vitro somatic membrane potential recordings from whole cell patch clamped rodent hippocampal CA1
neurons*, [doi:10.6080/K00P0WXQ](https://doi.org/10.6080/K00P0WXQ), streamed from
`https://crcnsarchive.s3-us-west-2.amazonaws.com/hc-10/live/Soltesz_data.zip`.

**Version.** CRCNS data sets are not versioned, and the paper cites no version (it predates
the data set). There is one published copy, under `hc-10/live/`; `hc-10/old/` holds only an
older description (`CRCNSAxoFiles.pdf`). The entry pins `crcns:hc-10/unversioned` and
identifies the files by the MD5 sums of the data set's `checksums.md5`: `Soltesz_data.zip`
99bc6c87c503c724a1e692f520e25162 (checked below while streaming), described by
`docs/crcns_hc-10_data_description.pdf` 75bb0f47568161037e46965f77f952c0 (version 0.7,
2 December 2016).

**Claims.** `claims/crcns-hc-10/10.1016-j.neuron.2014.03.034.yaml`, written from the version of
record with its Supplemental Information. The data set holds, for this paper, the current-step
sweeps of 8 superficial and 9 deep CA1 pyramidal cells (PCs) and 6 PV+ basket cells (PVBCs);
its other cells belong to other papers. Only the six claims on these recordings can be tested.

| Claim | Tested here |
|---|---|
| figs1e-if-curves | tested |
| figs1f-input-resistance | tested |
| figs1g-resting-potential | blocked (the sweeps are at the holding level); values shown |
| figs1h-sag | tested |
| tables1-rheobase | tested |
| figs3f-pvbc-firing | tested |
| fig1e to fig1n, fig2 to fig6, figs1-pc-morphology (26 claims) | blocked, data missing |
| fig4d-model-feedforward, fig4ef-model-pvbc-rates | blocked, too large (network model simulations) |

In [ ]:
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pingouin as pg
from scipy import stats
from statsmodels.formula.api import ols
from statsmodels.stats.anova import anova_lm

import hc10

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
t_start = time.time()

## Data check

The zip is streamed once through MD5 and compared with the data set's `checksums.md5`, then
opened in place. Each check of the description against the values is stated under its output.

In [ ]:
z = hc10.open_zip(check_md5=True)
print(f"zip MD5 matches checksums.md5: {hc10.ZIP_MD5}")
names = [n for n in z.namelist() if n.endswith(".atf")]
print(f"{len(names)} ATF files in the zip")
for info in z.infolist():
    if info.filename.endswith(".atf"):
        print(f"  {info.filename.split('/')[-1]:45s} {info.file_size:>10d} bytes")

In [ ]:
# Deposited more than once: identical bytes under two names.
import hashlib
digests = {}
for n in names:
    digests.setdefault(hashlib.md5(z.read(n)).hexdigest(), []).append(n.split("/")[-1])
dups = [v for v in digests.values() if len(v) > 1]
print(f"{len(digests)} distinct files among {len(names)}; duplicated: {dups}")

The descriptor's table lists 36 cells; the zip holds 37 files, because PV34 is deposited twice
(`PVBC_PV34.atf` and `PV34_PVBC_mouse.atf`, identical bytes). It is counted once.

In [ ]:
cells = {}
for group, members in [("dPC", hc10.PCS["dPC"]), ("sPC", hc10.PCS["sPC"]), ("PVBC", list(hc10.PVBCS))]:
    for c in members:
        d = hc10.load_cell(z, c)
        d["group"] = group
        d["on"], d["off"] = hc10.step_window(d["v"], d["steps"])
        cells[c] = d

rows = []
for c, d in cells.items():
    t, v = d["t"], d["v"]
    rows.append({
        "cell": c, "group": d["group"], "sweeps": v.shape[1],
        "steps_pA": f"{d['steps'][0]}:{d['steps'][1]-d['steps'][0]}:{d['steps'][-1]}",
        "samples": v.shape[0], "dt_ms": round(1000 * np.median(np.diff(t)), 4),
        "duration_s": round(t[-1] - t[0], 4),
        "signal": d["header"][-2].strip('"').split("=")[1],
        "step_on_s": round(t[d["on"]], 4), "step_len_s": round((d["off"] - d["on"]) / hc10.FS, 4),
        "v_min": round(v.min(), 1), "v_max": round(v.max(), 1),
    })
info = pd.DataFrame(rows)
print(info.to_string(index=False))
print()
print(info.groupby("group").size().rename("cells").to_string())
print("units of every voltage column:", sorted({t.split("(")[-1] for d in cells.values() for t in d["titles"][1:]}))
print("time column:", sorted({d["titles"][0] for d in cells.values()}))

Counts match the descriptor and the paper (sPC n = 8, dPC n = 9, Figure S1 legend), and each file
has one sweep per step of the descriptor's range (20 for the PCs, -400 to +550 pA; 13 to 23 for
the PVBCs). Every column is a voltage in mV; no file has a current column, so the step levels
come from the descriptor alone. Sampling is 0.1 ms (10 kHz, as the Methods state). Each sweep
lasts 2 s; the step starts at 0.5315 s and lasts 1.000 s in every cell, as the Supplemental
Experimental Procedures 2B ("1s-long step currents") and the descriptor ("about half a second
into the recording") say. Voltages range from -140 to +54 mV, as expected of a somatic
recording in mV.

The files have no trial table and only the voltage stream, so the check of the step labels is
made on the voltage itself: if sweep k carries the k-th step of the descriptor's range, the
steady-state deflection rises with k, is near 0 for the 0 pA sweep, and changes sign there.

In [ ]:
rows = []
for c, d in cells.items():
    base = hc10.baseline(d["v"], d["on"])
    dv = hc10.steady_state(d["v"], d["on"]) - base
    sub = d["steps"] <= 0  # below threshold, so no spike enters the steady state
    zero = d["steps"] == 0
    pre = d["v"][d["on"] - int(0.5 * hc10.FS):d["on"]]
    rows.append({
        "cell": c, "group": d["group"],
        "monotonic_up_to_0pA": bool(np.all(np.diff(dv[sub]) > 0)),
        "dV_at_0pA_mV": round(float(dv[zero][0]), 2),
        "dV_at_-100pA_mV": round(float(dv[d["steps"] == -100][0]), 2),
        "spearman_dV_vs_step": round(stats.spearmanr(d["steps"][sub], dv[sub])[0], 3),
        "baseline_0pA_mV": round(float(base[zero][0]), 2),
        "baseline_range_mV": f"{base.min():.1f} to {base.max():.1f}",
        "prestep_drift_sd_mV": round(float(pre.std(axis=0).max()), 2),
    })
chk = pd.DataFrame(rows)
print(chk.to_string(index=False))
print()
print("cells with monotonic subthreshold deflection:", int(chk.monotonic_up_to_0pA.sum()), "of", len(chk))
print("largest |dV| on the 0 pA sweep: %.2f mV" % chk["dV_at_0pA_mV"].abs().max())

In every PC the deflection rises with the sweep index through the subthreshold steps and is
within 0.4 mV of 0 on the sweep the descriptor gives as 0 pA, except D1_25abf (-1.61 mV), so
the sweep order is consistent with the descriptor's step ranges (consistent with, not shown:
the step amplitudes themselves are not recorded). Among the PVBCs only PV37 breaks the order,
between its -300 and -250 pA sweeps (-13.1 against -8.7 and -9.1 mV at -200 pA); its later
sweeps are in order. Before the step
every PC sits near -70 mV, the holding potential the Methods give ("the PCs were held at
-70mV"), and every PVBC near -60 mV, the interneuron holding potential of the Supplemental
Experimental Procedures ("holding potential was -60 mV"). The files carry no subject, slice,
region (septal or temporal) or soma position; groups come from the file names and the
descriptor's table.

## figs1e-if-curves

> Superficial and deep CA1 PCs do not differ in firing frequency as a function of injected
> current (two-way ANOVA, p > 0.4; 8 sPCs, 9 dPCs).

Steps 0 to +550 pA (Supplemental Experimental Procedures 2B), firing frequency under readings
(a) spikes in the 1 s step and (b) (spikes - 1) over the first-to-last spike interval, spikes as
upward crossings of 0 mV (the criterion). The paper's test is a two-way ANOVA of frequency on
group and step (here with their interaction, the standard two-way model); the mixed
(repeated-measures) ANOVA is the variant the criterion asks for. Expected at +550 pA: sPC
31.5 ± 3.15 Hz, dPC 30.5 ± 3.05 Hz.

In [ ]:
fi_rows = []
for c, d in cells.items():
    if d["group"] == "PVBC":
        continue
    n, ra, rb = hc10.firing(d, d["on"], d["off"])
    for s, k, a, b in zip(d["steps"], n, ra, rb):
        if s >= 0:
            fi_rows.append({"cell": c, "group": d["group"], "step": int(s), "spikes": int(k), "rate_a": a, "rate_b": b})
fi = pd.DataFrame(fi_rows)
tab = fi.pivot_table(index="step", columns="group", values=["rate_a", "rate_b"], aggfunc=["mean", "sem"])
print("Group mean and SEM of firing frequency (Hz) per step:")
print(tab.round(2).to_string())
print()
print("Per cell, reading (a), Hz:")
print(fi.pivot_table(index="cell", columns="step", values="rate_a").round(0).astype(int).to_string())

In [ ]:
def anova_group(df, col):
    m = ols(f"{col} ~ C(group) * C(step)", data=df).fit()
    full = anova_lm(m, typ=2)
    add = anova_lm(ols(f"{col} ~ C(group) + C(step)", data=df).fit(), typ=2)
    mixed = pg.mixed_anova(data=df, dv=col, within="step", between="group", subject="cell")
    return full, add, mixed

fi_results = {}
for col in ["rate_a", "rate_b"]:
    full, add, mixed = anova_group(fi, col)
    g = full.loc["C(group)"]
    eta = g["sum_sq"] / full["sum_sq"].sum()
    mg = mixed.set_index("Source").loc["group"]
    fi_results[col] = dict(F=g["F"], p=g["PR(>F)"], eta=eta, p_add=add.loc["C(group)", "PR(>F)"],
                           p_inter=full.loc["C(group):C(step)", "PR(>F)"], F_mixed=mg["F"], p_mixed=mg["p_unc"],
                           np2_mixed=mg["np2"])
    print(f"Reading {col[-1]}: two-way ANOVA (group x step): group F = {g['F']:.3f}, p = {g['PR(>F)']:.3f}, "
          f"eta^2 = {eta:.4f}; interaction p = {fi_results[col]['p_inter']:.3f}; additive model group p = "
          f"{fi_results[col]['p_add']:.3f}")
    print(f"           mixed ANOVA (step within cell): group F = {mg['F']:.3f}, p = {mg["p_unc"]:.3f}, "
          f"partial eta^2 = {mg['np2']:.3f}")

at550 = fi[fi.step == 550].groupby("group")[["rate_a", "rate_b"]].agg(["mean", "sem"])
print()
print("At +550 pA (Hz):")
print(at550.round(2).to_string())
exp = {"sPC": (31.5, 3.15), "dPC": (30.5, 3.05)}
for col in ["rate_a", "rate_b"]:
    ok = {g: abs(at550.loc[g, (col, "mean")] - v) <= tol for g, (v, tol) in exp.items()}
    print(f"reading {col[-1]}: sPC within 31.5 +- 3.15: {ok['sPC']}; dPC within 30.5 +- 3.05: {ok['dPC']}")

In [ ]:
# Spike threshold: the 0 mV crossing is a standard choice; counts with -20 mV and +10 mV.
alt = []
for thr in [-20.0, 0.0, 10.0]:
    tot = 0
    for c, d in cells.items():
        if d["group"] == "PVBC":
            continue
        seg = d["v"][d["on"]:d["off"]]
        tot += int(((seg[1:] >= thr) & (seg[:-1] < thr)).sum())
    alt.append((thr, tot))
print("total spikes in all PC steps by threshold (mV):", alt)

fig, ax = plt.subplots(1, 2, figsize=(9, 3.2))
for i, col in enumerate(["rate_a", "rate_b"]):
    for g, colr in [("sPC", "tab:blue"), ("dPC", "tab:green")]:
        s = fi[fi.group == g].groupby("step")[col]
        ax[i].errorbar(s.mean().index, s.mean(), s.sem(), color=colr, label=g, marker="o", ms=3)
    ax[i].set(xlabel="step (pA)", ylabel="firing frequency (Hz)", title=f"reading ({col[-1]})")
    ax[i].legend()
plt.tight_layout(); plt.show()
print("plotted range, reading a: %.1f to %.1f Hz; reading b: %.1f to %.1f Hz (all points within the axes)"
      % (fi.rate_a.min(), fi.rate_a.max(), fi.rate_b.min(), fi.rate_b.max()))

**Result: reproduced.** Under both readings the two-way ANOVA finds no group effect (reading a:
F = 0.68, p = 0.411, eta^2 = 0.0005; reading b: p = 0.866), nor does the mixed ANOVA (p = 0.759
and 0.951), and the group means at +550 pA are within tolerance (reading a: sPC 31.62, dPC 30.56
Hz; reading b: 33.63 and 31.24 Hz). Reading (a) also gives the paper's p > 0.4.

## figs1f-input-resistance

> Superficial and deep PCs do not differ in input resistance (117 against 124 MOhm).

Steady state 0.8 to 1.0 s after step onset on the -100 to +100 pA steps (Supplemental
Experimental Procedures 2B). Readings (a) slope of a linear fit over the five steps, (b) mean of
change over current over the four non-zero steps. Group test by the paper's rule (Shapiro-Wilk
in each group; two-tailed unpaired t test, else Mann-Whitney).

In [ ]:
def paper_rule(x, y, paired=False):
    # Statistical Analysis: t test unless a group fails Shapiro-Wilk, then the rank test.
    normal = stats.shapiro(x).pvalue >= 0.05 and stats.shapiro(y).pvalue >= 0.05
    if paired:
        return ("paired t", stats.ttest_rel(x, y).pvalue) if normal else ("Wilcoxon", stats.wilcoxon(x, y).pvalue)
    return ("t", stats.ttest_ind(x, y).pvalue) if normal else ("Mann-Whitney", stats.mannwhitneyu(x, y).pvalue)

def cohen_d(x, y):
    sp = np.sqrt(((len(x) - 1) * np.var(x, ddof=1) + (len(y) - 1) * np.var(y, ddof=1)) / (len(x) + len(y) - 2))
    return (np.mean(x) - np.mean(y)) / sp

ri_rows = []
for c, d in cells.items():
    if d["group"] == "PVBC":
        continue
    a, b = hc10.input_resistance(d, d["on"])
    n, _, _ = hc10.firing(d, d["on"], d["off"])
    # Variant: only the steps without spikes (-100 to 0 pA in every cell).
    sel = (d["steps"] >= -100) & (d["steps"] <= 0)
    ss = hc10.steady_state(d["v"], d["on"])[sel]
    a_sub = np.polyfit(d["steps"][sel], ss, 1)[0] * 1000
    ri_rows.append({"cell": c, "group": d["group"], "Rin_a": a, "Rin_b": b, "Rin_sub": a_sub,
                    "spikes_at_+100": int(n[d["steps"] == 100][0])})
ri = pd.DataFrame(ri_rows)
print(ri.round(1).to_string(index=False))
print()
exp = {"sPC": (117, 11.7), "dPC": (124, 12.4)}
for col in ["Rin_a", "Rin_b", "Rin_sub"]:
    s, dd = ri[ri.group == "sPC"][col].values, ri[ri.group == "dPC"][col].values
    test, p = paper_rule(s, dd)
    ok = [abs(np.mean(x) - exp[g][0]) <= exp[g][1] for g, x in [("sPC", s), ("dPC", dd)]]
    print(f"{col}: sPC {s.mean():.1f} +- {stats.sem(s):.1f}, dPC {dd.mean():.1f} +- {stats.sem(dd):.1f} MOhm; "
          f"{test} p = {p:.3f}; Cohen's d (sPC - dPC) = {cohen_d(s, dd):.2f}; within tolerance sPC {ok[0]}, dPC {ok[1]}")

**Result: reproduced.** Under reading (a) sPC 118.7 and dPC 124.9 MOhm, both within tolerance of
117 and 124, with no group difference (Shapiro-Wilk fails, Mann-Whitney p = 1.000, Cohen's d =
-0.13); reading (b) gives 118.9 and 123.7 MOhm (p = 1.000). Three cells (D1_25abf, D4_11abf,
S1_47abf) fire on the +100 pA step, which enters the steady state; with only the -100 to 0 pA
steps, sPC 107.2 and dPC 101.8 MOhm (p = 0.673), the dPC mean then outside tolerance.

## figs1g-resting-potential (blocked)

> Superficial PCs have more depolarized resting membrane potentials than deep PCs (about -66
> against -72 mV uncorrected; -80.9 against -86.6 mV corrected for the junction potential).

The criterion reads the resting potential as the pre-step baseline of sweeps with zero holding
current, and says that if the files have no current channel and a baseline at the holding level,
the sweeps do not hold the resting potential and the claim cannot be tested on them. The
values are shown so that this can be seen.

In [ ]:
rest = chk[chk.group != "PVBC"][["cell", "group", "baseline_0pA_mV", "baseline_range_mV"]]
print(rest.to_string(index=False))
s = rest[rest.group == "sPC"].baseline_0pA_mV.values
dd = rest[rest.group == "dPC"].baseline_0pA_mV.values
test, p = paper_rule(s, dd)
print(f"pre-step baseline of the 0 pA sweep: sPC {s.mean():.2f} +- {stats.sem(s):.2f}, dPC {dd.mean():.2f} +- "
      f"{stats.sem(dd):.2f} mV; {test} p = {p:.3f}")
allpc = np.concatenate([s, dd])
print(f"all 17 PCs: {allpc.min():.1f} to {allpc.max():.1f} mV, mean {allpc.mean():.2f}; distance from the -70 mV "
      f"holding level at most {np.abs(allpc + 70).max():.1f} mV")
print("current channel in any PC file:", any("pA" in t for d in cells.values() if d["group"] != "PVBC" for t in d["titles"]))

**Result: blocked (data missing).** No PC file has a current channel, and every PC's pre-step
baseline is within 2.6 mV of the -70 mV holding potential (sPC -70.03, dPC -69.76 mV), so by the
criterion the sweeps do not hold the resting potential. The Table S1 footnote says the tonic
current was not recorded.

## figs1h-sag

> Superficial PCs show larger sag potentials than deep PCs in response to large hyperpolarizing
> current pulses (about 8.7 against 5.6 mV).

Sag is steady state (0.8 to 1.0 s) minus the most negative voltage in the step. Readings for the
step: (a) -400, (b) -300, (c) -200 pA (the criterion). Expected sPC 8.7 ± 0.87, dPC 5.6 ± 0.56 mV.

In [ ]:
sag_rows = []
for c, d in cells.items():
    if d["group"] == "PVBC":
        continue
    row = {"cell": c, "group": d["group"]}
    for st in [-400, -300, -200]:
        row[f"sag_{st}"] = hc10.sag(d, d["on"], st)
        # Variant: peak from the voltage smoothed over 1 ms, against noise in the minimum.
        k = int(np.where(d["steps"] == st)[0][0])
        x = np.convolve(d["v"][:, k], np.ones(10) / 10, mode="same")
        row[f"sag_{st}_smooth"] = hc10.steady_state(d["v"][:, [k]], d["on"])[0] - x[d["on"]:d["on"] + 10000].min()
    sag_rows.append(row)
sg = pd.DataFrame(sag_rows)
print(sg.round(2).to_string(index=False))
print()
exp = {"sPC": (8.7, 0.87), "dPC": (5.6, 0.56)}
for col in [c for c in sg.columns if c.startswith("sag_")]:
    s, dd = sg[sg.group == "sPC"][col].values, sg[sg.group == "dPC"][col].values
    test, p = paper_rule(s, dd)
    ok = [abs(np.mean(x) - exp[g][0]) <= exp[g][1] for g, x in [("sPC", s), ("dPC", dd)]]
    print(f"{col:17s}: sPC {s.mean():.2f} +- {stats.sem(s):.2f}, dPC {dd.mean():.2f} +- {stats.sem(dd):.2f} mV; "
          f"{test} p = {p:.4f}; d = {cohen_d(s, dd):.2f}; within tolerance sPC {ok[0]}, dPC {ok[1]}")

**Result: reproduced.** Under reading (a), the -400 pA step, sPC sag is 8.71 mV and dPC 5.69 mV,
both within tolerance, and sPCs exceed dPCs (t test p = 0.0020, Cohen's d = 1.81). Readings (b)
and (c) give the same direction and significance (p = 0.0011 and 0.0050) with smaller values,
outside tolerance (-300 pA: 7.04 and 4.48 mV; -200 pA: 5.25 and 3.56 mV). Smoothing the voltage
over 1 ms before taking the peak changes the means by at most 0.05 mV.

## tables1-rheobase

> The experimental superficial and deep PCs have the same rheobase, +150 pA, at the 50 pA
> resolution of the step protocol.

Per cell, the smallest step with at least one spike (steps are relative to the holding current,
so the tonic current is excluded, as the Table S1 footnote says). Readings for the group: (a)
median of per-cell rheobases, (b) mean rounded to the nearest 50 pA step.

In [ ]:
rb_rows = []
for c, d in cells.items():
    if d["group"] == "PVBC":
        continue
    n, _, _ = hc10.firing(d, d["on"], d["off"])
    rb_rows.append({"cell": c, "group": d["group"], "rheobase_pA": int(d["steps"][np.argmax(n > 0)]),
                    "spikes_at_rheobase": int(n[np.argmax(n > 0)])})
rbd = pd.DataFrame(rb_rows)
print(rbd.to_string(index=False))
print()
for g in ["sPC", "dPC"]:
    x = rbd[rbd.group == g].rheobase_pA.values
    print(f"{g}: per-cell rheobases {sorted(int(v) for v in x)}; (a) median {np.median(x):.0f} pA; "
          f"(b) mean {x.mean():.1f} pA, rounded to 50 pA: {50 * np.round(x.mean() / 50):.0f} pA")
s, dd = rbd[rbd.group == "sPC"].rheobase_pA.values, rbd[rbd.group == "dPC"].rheobase_pA.values
print(f"groups compared (paper's rule): {paper_rule(s, dd)[0]} p = {paper_rule(s, dd)[1]:.3f}")

**Result: partially reproduced.** Reading (a) gives dPC 150 pA, as Table S1, but sPC 200 pA;
reading (b) gives the same step for both groups, 200 pA (means 187.5 and 177.8 pA), not 150. By
the criterion this is partial: the groups share a step under (b), but it is not +150 pA. The
per-cell rheobases do not differ between groups (t test p = 0.732).

## figs3f-pvbc-firing

> A biological PV+ basket cell does not fire up to +300 pA, fires at about 2 Hz at +350 pA, and
> its firing rises steeply above that, to about 17 Hz at +400, 36 Hz at +450 and 52 Hz at +500 pA,
> matched by the PVBC model.

Firing per step as in figs1e-if-curves (readings a and b of the rate). Readings for the cell:
(a) any one PVBC, (b) the mean over PVBCs on the steps every cell has. The 1 s step of 50 pA
increments is the claim's stated assumption; the data show a 1 s step and the descriptor gives
50 pA increments for all six PVBCs.

In [ ]:
pv_rows = []
for c, d in cells.items():
    if d["group"] != "PVBC":
        continue
    n, ra, rb = hc10.firing(d, d["on"], d["off"])
    for s, k, a, b in zip(d["steps"], n, ra, rb):
        if s >= 0:
            pv_rows.append({"cell": c, "step": int(s), "spikes": int(k), "rate_a": a, "rate_b": b})
pv = pd.DataFrame(pv_rows)
print("Reading (a) of the rate, Hz, per cell and step (blank: step not recorded):")
print(pv.pivot_table(index="cell", columns="step", values="rate_a").round(1).to_string(na_rep=""))
print()
print("Reading (b) of the rate, Hz:")
print(pv.pivot_table(index="cell", columns="step", values="rate_b").round(1).to_string(na_rep=""))

targets = {350: (2, 1), 400: (17, 1.7), 450: (36, 3.6), 500: (52, 5.2)}
print()
for c in hc10.PVBCS:
    p = pv[pv.cell == c].set_index("step")
    silent = bool((p.loc[0:300, "spikes"] == 0).all())
    for col in ["rate_a", "rate_b"]:
        vals = {s: (p.loc[s, col] if s in p.index else np.nan) for s in targets}
        within = all(s in p.index and abs(vals[s] - v) <= tol for s, (v, tol) in targets.items())
        print(f"{c} {col}: silent 0..+300 pA: {silent}; at 350/400/450/500: "
              + ", ".join("--" if np.isnan(x) else f"{x:.1f}" for x in vals.values())
              + f"; all four within tolerance: {within}")

common = pv.groupby("step").cell.nunique()
common = common[common == len(hc10.PVBCS)].index
print()
print(f"reading (b): steps every PVBC has: 0 to {common.max()} pA; mean rate_a there:",
      pv[pv.step.isin(common)].groupby("step").rate_a.mean().round(2).to_dict())
print("mean over the PVBCs that have each step (variant):",
      pv.groupby("step").agg(n=("cell", "nunique"), rate_a=("rate_a", "mean")).loc[[350, 400, 450, 500]].round(1).to_dict("index"))

**Result: reproduced.** Under reading (a) of the cell and of the rate (spikes per second), PVBC
PV34 is silent from 0 to +300 pA and fires at 2, 17, 36 and 52 Hz at +350, +400, +450 and +500
pA, the four expected values. With the rate's reading (b) PV34 gives 14.9 Hz at +350 pA (two
spikes), outside tolerance. No other PVBC matches; PV37, PV38, PV72 and PV80 fire at or below
+300 pA, and PV36 starts at +500 pA. The mean of reading (b) of the cell cannot be formed above
+300 pA, since PV38 was recorded only to +300 pA (mean 16.17 Hz at +300 pA). The model's half of
the statement is not tested.

## Claims not tested

The other 28 claims rest on data the data set does not hold: paired recordings (Figures 1, 3, 5,
6), anatomy and bouton counts (Figures 1J to 1L, 2G to 2I, 5, S1A to S1D, S4, S5),
optogenetics (Figure 1N), in vivo two-photon imaging (Figure 2B to 2F, S2), PVBC trains against PC
firing (Figure 4A, 4B), retrograde labelling (Figure 6), and the network model's simulations
(Figure 4C to 4F). The two model claims could use the model's public code (ModelDB 153280), but
its simulations are far too large to run here: a 20 ms run of one network, timed while preparing
this entry and not run here, took 13.5 minutes on one core, so the 39 runs of 5 s the claims
need are estimated at about 670 core-hours. The zip's other files are axo-axonic, bistratified, CCK+ basket, ivy,
neurogliaform and Schaffer collateral-associated cells from other papers or unpublished.

In [ ]:
other = sorted({n.split("/")[-1] for n in names} - {hc10.member_name(names, c).split("/")[-1] for c in cells}
               - {"PV34_PVBC_mouse.atf"})
print(f"{len(other)} files not used:", other)
print(f"notebook run time: {(time.time() - t_start) / 60:.1f} min")

## Summary

| Claim | Outcome |
|---|---|
| figs1e-if-curves | reproduced |
| figs1f-input-resistance | reproduced |
| figs1g-resting-potential | blocked: data missing (sweeps at the holding potential) |
| figs1h-sag | reproduced |
| tables1-rheobase | partially reproduced |
| figs3f-pvbc-firing | reproduced |
| 26 claims on paired recordings, anatomy, imaging and optogenetics | blocked: data missing |
| 2 claims on the network model | blocked: too large |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

Differences from the paper's method: the step amplitudes come from the descriptor, since the
files record no current; firing frequency, input resistance, sag and rheobase use the readings
the claims file lists, and spikes are counted at 0 mV; the paper's ANOVA for Figure S1E is run
with and without the interaction and as a mixed ANOVA.

What was found about the data set: PV34 is deposited twice under two names; the files record no
injected current, no holding current, and no animal, slice, region or soma position for any cell;
the PC sweeps start at the -70 mV holding potential, so the resting potential of Figure S1G and
Table S1 is not in them; the step timing and the sweep order agree with the descriptor.